# core

> Provision Modal GPU sandboxes and wire them up as remote IPython kernels for solveit.

In [ ]:
#| default_exp core

In [ ]:
#| export
from solveit_modal.modal import *
from solveit_modal.ipyfernel import *
from ipyfernel.core import *

In [ ]:
#| export
def provision_sandbox(gpu='T4',   # GPU type string, e.g. 'T4', 'A100'
                      timeout=600 # Max sandbox lifetime in seconds
                     ) -> tuple:  # (sandbox, host, port)
    "Create Modal app, image, sandbox, and expose TCP tunnel."
    app = mk_app()
    img = mk_img()
    sb = mk_sandbox(app, img, gpu=gpu, timeout=timeout)
    host, port = get_tunnel(sb)
    return sb, host, port

In [ ]:
#| export
def setup_ssh(sb,    # Modal Sandbox object
              host,  # Tunnel hostname
              port   # Tunnel port
             ) -> object:  # Paramiko SSH connection
    "Inject public key, start SSH service, and verify connectivity."
    inject_pubkey(sb, get_pubkey())
    start_ssh(sb)
    ssh = mk_ssh(host, port)
    verify_ssh(ssh)
    return ssh

In [ ]:
#| export
def link_remote_kernel(ssh,   # Paramiko SSH connection
                       host,  # Tunnel hostname
                       port   # Tunnel port
                      ) -> None:
    "Launch remote IPython kernel on sandbox and connect ipyfernel to it."
    start_remote_kernel(ssh)
    p = get_remote_python(ssh)
    register_remote_kernel(remote_python=p)
    connect_existing_kernel(f'{host}:{port}')
    set_sticky()

In [ ]:
#| export
def gpu_on(gpu='T4',   # GPU type string
           timeout=600 # Max sandbox lifetime in seconds
          ) -> tuple:  # (sandbox, ssh)
    "Provision a GPU sandbox, wire up SSH, and hijack cells onto a remote kernel."
    sb, host, port = provision_sandbox(gpu, timeout)
    ssh = setup_ssh(sb, host, port)
    link_remote_kernel(ssh, host, port)
    return sb, ssh

In [ ]:
#| export
def gpu_off(sb  # sandbox object from gpu_on
          ) -> None:
    "Restore local kernel, disconnect remote, and teardown sandbox. Must be called from a %%local cell."
    unset_sticky()
    stop_remote()
    sb.terminate()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()